In [1]:
import pandas as pd
import io
import os

def extract_reactor_time_series(file_path):
    """
    Robust stream parser designed for non-tabular IAEA/NEA performance reports.
    Manually skips unstructured metadata blocks to isolate time-series headers.
    """
    # Dynamically extract the reactor unit name from the filename for professional logging
    base_name = os.path.basename(file_path).replace('_', ' ').replace('.csv', '').title()
    
    with open(file_path, 'r', encoding='utf-8-sig') as f:
        lines = f.readlines()

    # 1. Locate the dynamic table anchor point
    start_idx = -1
    for i, line in enumerate(lines):
        if line.startswith("Year,Month,EnergyGWh"):
            start_idx = i
            break

    if start_idx == -1:
        raise ValueError(f"Structural Violation: Missing expected time-series headers in {base_name}.")

    # 2. Extract the specific table block cleanly via stream inspection
    isolated_table_lines = []
    for line in lines[start_idx:]:
        if line.startswith("Energy_label") or line.strip() == "":
            break
        isolated_table_lines.append(line)

    # 3. Stream data directly into an isolated Pandas memory buffer
    csv_stream = "".join(isolated_table_lines)
    df = pd.read_csv(io.StringIO(csv_stream))

    # 4. Standard Database Cleaning & Casting Pipeline
    # Ensure all string column spaces are stripped out 
    if 'Month' in df.columns:
        df['Month'] = df['Month'].astype(str).str.strip()
        
    # Standardize string-formatted currency/metrics commas into pure floats
    if df['EnergyGWh'].dtype == 'object':
        df['EnergyGWh'] = df['EnergyGWh'].str.replace(',', '', regex=False).astype(float)

    print(f"✅ Telemetry Successfully Extracted: {base_name} Model Pipeline Ready.")
    return df

# --- Execution ---
file_path = './data/barakah_1.csv'

try:
    production_df = extract_reactor_time_series(file_path)
    
    print("\n--- Raw Head View (Metrics Extraction Validate) ---")
    print(production_df.head(12))

    print("\n--- Operational Window Check ---")
    print(production_df[['Month', 'EnergyGWh', 'PUF_PlannedUnavailabilityFactor', 'FLR_ForcedLossRate']])

except Exception as e:
    print(f"❌ Pipeline Execution Failure: {e}")


❌ Pipeline Execution Failure: [Errno 2] No such file or directory: './data/barakah_1.csv'
